In [1]:
# Most code copied from reference 1 and reference 2
!python -m pip install numpy
!python -m pip install matplotlib
!python -m pip install pandas
!python -m pip install scikit-learn

In [2]:
# Load necessary modules and libraries

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import learning_curve
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic, Matern, ExpSineSquared,DotProduct

import time
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io
from google.colab import files
uploaded = files.upload() # "Surface_features.csv"
Dataset = files.upload()  # "CNN_Data.xlsx"

Saving Surface_features.csv to Surface_features.csv


Saving CNN_Data.xlsx to CNN_Data.xlsx


In [ ]:
Inputs = pd.read_excel(io.BytesIO(Dataset['CNN_Data.xlsx']),header=0).values
for iii in range(len(Inputs)):
  # Set number of qubits/wires
  a = int(Inputs[iii,1])
  b = int(Inputs[iii,2])
  c = int(Inputs[iii,3])
  Perc_Train = Inputs[iii,4]
  print("dataset", iii)
  Geometry1 = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0, usecols=(4,8,9,10,11,12,14)).iloc[0:45]
  Geometry = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0, usecols=(4,6,7,8,9,10,11,12)).iloc[0:45].values

  Ra_ch = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(5,)).iloc[0:45].values
  Ra_ch = Ra_ch[:,0]
  ks = pd.read_csv(io.BytesIO(uploaded['Surface_features.csv']),header=0,usecols=(13,)).iloc[0:45].values
  ks = ks[:,0]



  Geometry1["ks"]= np.divide(ks,Ra_ch)
  Geometry1["krms_ch"]= np.divide(Geometry1["krms_ch"],Ra_ch)


  Geometry1.rename({'krms_ch': '$k_{rms}/R_a$',
                  'pro_ch': '$P_o$',
                  'ESx_ch': '$E_x$',
                  'ESz_ch': '$E_z$',
                  'sk_ch': '$S_k$',
                  'ku_ch': '$K_u$',
                  'ks': '$k_s/R_a$',
                  'label': 'Label',
                  }, axis='columns', errors="raise",inplace = True)
  # Data reconfiguration, to be used in ML

  X = Geometry
  y = np.divide(ks,Ra_ch)


  X[:,0] = np.divide(X[:,0],Ra_ch)
  X[:,2] = np.abs(X[:,2])


  # Generate secondary features and append them to the original dataset

  n,m = X.shape
  X0 = np.ones((n,1))
  X1 = np.ones((n,1))
  X2 = np.ones((n,1))
  X3 = np.ones((n,1))
  X4 = np.ones((n,1))
  X5 = np.ones((n,1))
  X6 = np.ones((n,1))
  X7 = np.ones((n,1))
  X8 = np.ones((n,1))
  X9 = np.ones((n,1))




  X1[:,0] = np.transpose(X[:,4]*X[:,5])
  X2[:,0] = np.transpose(X[:,4]*X[:,6])
  X3[:,0] = np.transpose(X[:,4]*X[:,7])
  X4[:,0] = np.transpose(X[:,5]*X[:,6])
  X5[:,0] = np.transpose(X[:,5]*X[:,7])
  X6[:,0] = np.transpose(X[:,6]*X[:,7])
  X7[:,0] = np.transpose(X[:,4]*X[:,4])
  X8[:,0] = np.transpose(X[:,5]*X[:,5])
  X9[:,0] = np.transpose(X[:,6]*X[:,6])




  X = np.hstack((X,X1))
  X = np.hstack((X,X2))
  X = np.hstack((X,X3))
  X = np.hstack((X,X4))
  X = np.hstack((X,X5))
  X = np.hstack((X,X6))
  X = np.hstack((X,X7))
  X = np.hstack((X,X8))
  X = np.hstack((X,X9))
  # Best linear estimation


  reg = LinearRegression().fit(X, y)
  reg.score(X, y)
  yn=reg.predict(X)
  print("Mean err: %f" % np.mean(100.*abs(yn-y)/(y)))
  print("Max err: %f" % max(100.*abs(yn-y)/(y)))

  # Define three files that store the best ML prediction based on either L_\infty, L_1 and a custom L norms

  filename1 = 'MLP_Linf.sav'
  filename2 = 'MLP_L1.sav'
  filename3 = 'MLP_L.sav'
  Layers = np.array([a,b,c])
  N_of_Param = 17*Layers[0]+Layers[0]+Layers[0]*Layers[1]+Layers[1]+Layers[1]*Layers[2]+Layers[2]+Layers[2]*1+1
  print("Number of parameters: %f" % N_of_Param)
  # Perform ML training --- it may take some time.
  # Adjust ranges for by1 through by4 for faster (but potentially less accurate) results.



  miny1=100
  miny2=50
  miny3=1
  start_time = time.perf_counter()
  for by1 in [a]: #18
      for by2 in [b]: #7
          for by3 in [c]: #7

              my_mlp_obj_1 = MLPRegressor(hidden_layer_sizes=(by1,by2,by3), max_iter=200000, alpha=1,
                    solver='lbfgs', verbose=0, tol=1e-5, activation = 'relu', learning_rate_init=.01)
              by4=0.
              print("by1: %f" % by1)
              print("by2: %f" % by2)
              print("by3: %f" % by3)

              while by4<100.:
                  by4=by4+1
                  print("Outer Iteration: %f" % by4)

                  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=Perc_Train)
                  my_mlp_obj_1.fit(X_train, y_train)
                  yn=my_mlp_obj_1.predict(X)


                  if miny1>max(100.*abs(yn-y)/y):
                      pickle.dump(my_mlp_obj_1, open(filename1, 'wb'))
                      miny1=max(100.*abs(yn-y)/y)
                      print("Miny1: %f" % miny1)

                  if miny2>np.mean(100.*abs(yn-y)/y):
                      pickle.dump(my_mlp_obj_1, open(filename2, 'wb'))
                      miny2=np.mean(100.*abs(yn-y)/y)
                      print("Miny2: %f" % miny2)

                  if miny3>max(abs(np.log(yn/y))):
                      pickle.dump(my_mlp_obj_1, open(filename3, 'wb'))
                      miny3=max(abs(np.log(yn/y)))
                      print("Miny3: %f" % miny3)


  end_time = time.perf_counter()
  execution_time = end_time - start_time
  print("by1: %f" % by1)
  print("by2: %f" % by2)
  print("by3: %f" % by3)
  loaded_model = pickle.load(open(filename2, 'rb'))
  yn = loaded_model.predict(X)

  # Write data to file
  df_to_insert = pd.DataFrame({"Value": [N_of_Param]})
  with pd.ExcelWriter("CNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=5, header=False, index=False)
  df_to_insert = pd.DataFrame({"Value": [np.mean(100.*abs(yn-y)/(y))]})
  with pd.ExcelWriter("CNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=6, header=False, index=False)
  df_to_insert = pd.DataFrame({"Value": [max(100.*abs(yn-y)/(y))]})
  with pd.ExcelWriter("CNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=7, header=False, index=False)
  df_to_insert = pd.DataFrame({"Value": [execution_time]})
  with pd.ExcelWriter("CNN_Data.xlsx", engine="openpyxl",mode="a",if_sheet_exists="overlay",) as writer: df_to_insert.to_excel( writer, sheet_name="Sheet1", startrow=iii+1, startcol=8, header=False, index=False)
files.download('CNN_Data.xlsx')

dataset 0
Mean err: 14.945903
Max err: 51.586233
Number of parameters: 51.000000
by1: 2.000000
by2: 2.000000
by3: 2.000000
Outer Iteration: 1.000000
Outer Iteration: 2.000000
Miny1: 80.733927
Miny2: 12.682838
Miny3: 0.591856
Outer Iteration: 3.000000
Outer Iteration: 4.000000
Miny1: 45.716050
Miny2: 10.604711
Miny3: 0.402180
Outer Iteration: 5.000000
Outer Iteration: 6.000000
Outer Iteration: 7.000000
Outer Iteration: 8.000000
Outer Iteration: 9.000000
Outer Iteration: 10.000000
Outer Iteration: 11.000000
Outer Iteration: 12.000000
Outer Iteration: 13.000000
Miny1: 38.051549
Outer Iteration: 14.000000
Outer Iteration: 15.000000
Outer Iteration: 16.000000
Outer Iteration: 17.000000
Outer Iteration: 18.000000
Outer Iteration: 19.000000
Outer Iteration: 20.000000
Miny3: 0.345872
Outer Iteration: 21.000000
Outer Iteration: 22.000000
Outer Iteration: 23.000000
Outer Iteration: 24.000000
Outer Iteration: 25.000000
Outer Iteration: 26.000000
Outer Iteration: 27.000000
Outer Iteration: 28.0000

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL: .

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Miny2: 20.359978
Miny3: 0.883489
Outer Iteration: 3.000000
Outer Iteration: 4.000000
Miny1: 73.419009
Miny2: 14.306769
Miny3: 0.550540
Outer Iteration: 5.000000
Outer Iteration: 6.000000
Outer Iteration: 7.000000
Outer Iteration: 8.000000
Outer Iteration: 9.000000
Miny1: 51.404527
Miny2: 12.744103
Miny3: 0.414785
Outer Iteration: 10.000000
Outer Iteration: 11.000000
Outer Iteration: 12.000000
Outer Iteration: 13.000000
Outer Iteration: 14.000000
Outer Iteration: 15.000000
Outer Iteration: 16.000000
Outer Iteration: 17.000000
Outer Iteration: 18.000000
Outer Iteration: 19.000000
Outer Iteration: 20.000000
Miny2: 11.039659
Outer Iteration: 21.000000
Outer Iteration: 22.000000
Outer Iteration: 23.000000
Outer Iteration: 24.000000
Outer Iteration: 25.000000
Outer Iteration: 26.000000
Outer Iteration: 27.000000
Outer Iteration: 28.000000
Outer Iteration: 29.000000
Outer Iteration: 30.000000
Outer Iteration: 31.000000
Outer Iteration: 32.000000
Outer Iteration: 33.000000
Outer Iteration: 34.

/tmp/ipykernel_6216/2934069003.py:137: RuntimeWarning: invalid value encountered in log
  if miny3>max(abs(np.log(yn/y))):


Outer Iteration: 23.000000
Outer Iteration: 24.000000
Outer Iteration: 25.000000
Outer Iteration: 26.000000
Outer Iteration: 27.000000
Outer Iteration: 28.000000
Outer Iteration: 29.000000
Outer Iteration: 30.000000
Outer Iteration: 31.000000
Outer Iteration: 32.000000
Outer Iteration: 33.000000
Outer Iteration: 34.000000
Outer Iteration: 35.000000
Miny1: 61.361120
Miny3: 0.530261
Outer Iteration: 36.000000
Outer Iteration: 37.000000
Outer Iteration: 38.000000
Outer Iteration: 39.000000
Outer Iteration: 40.000000
Miny1: 58.006359
Miny3: 0.468579
Outer Iteration: 41.000000
Outer Iteration: 42.000000
Outer Iteration: 43.000000
Outer Iteration: 44.000000
Outer Iteration: 45.000000
Outer Iteration: 46.000000
Outer Iteration: 47.000000
Outer Iteration: 48.000000
Outer Iteration: 49.000000
Outer Iteration: 50.000000
Outer Iteration: 51.000000
Outer Iteration: 52.000000
Outer Iteration: 53.000000
Outer Iteration: 54.000000
Outer Iteration: 55.000000
Outer Iteration: 56.000000
Outer Iteration:

/tmp/ipykernel_6216/2934069003.py:137: RuntimeWarning: invalid value encountered in log
  if miny3>max(abs(np.log(yn/y))):


... (15 outputs left out)

/tmp/ipykernel_6216/2934069003.py:137: RuntimeWarning: invalid value encountered in log
  if miny3>max(abs(np.log(yn/y))):


Outer Iteration: 43.000000
Outer Iteration: 44.000000
Outer Iteration: 45.000000
Outer Iteration: 46.000000
Outer Iteration: 47.000000
Outer Iteration: 48.000000
Outer Iteration: 49.000000
Outer Iteration: 50.000000
Outer Iteration: 51.000000
Outer Iteration: 52.000000
Outer Iteration: 53.000000
Outer Iteration: 54.000000
Outer Iteration: 55.000000
Outer Iteration: 56.000000
Outer Iteration: 57.000000
Outer Iteration: 58.000000
Outer Iteration: 59.000000
Outer Iteration: 60.000000
Outer Iteration: 61.000000
Outer Iteration: 62.000000
Outer Iteration: 63.000000
Outer Iteration: 64.000000
Outer Iteration: 65.000000
Outer Iteration: 66.000000
Outer Iteration: 67.000000
Outer Iteration: 68.000000
Outer Iteration: 69.000000
Outer Iteration: 70.000000
Outer Iteration: 71.000000
Outer Iteration: 72.000000
Outer Iteration: 73.000000
Outer Iteration: 74.000000
Outer Iteration: 75.000000
Outer Iteration: 76.000000
Outer Iteration: 77.000000
Outer Iteration: 78.000000
Outer Iteration: 79.000000
O

/tmp/ipykernel_6216/2934069003.py:137: RuntimeWarning: invalid value encountered in log
  if miny3>max(abs(np.log(yn/y))):


Outer Iteration: 11.000000
Outer Iteration: 12.000000
Outer Iteration: 13.000000
Outer Iteration: 14.000000
Outer Iteration: 15.000000
Outer Iteration: 16.000000
Outer Iteration: 17.000000
Outer Iteration: 18.000000
Outer Iteration: 19.000000
